# Broadcasting

## Definition
**Broadcasting** is NumPy's mechanism for performing operations on arrays of **different shapes**
without making explicit copies. NumPy automatically expands (broadcasts) smaller arrays to match
the shape of larger ones — conceptually, not in actual memory.

## Broadcasting Rules (applied axis by axis, right-to-left):
1. If arrays have different numbers of dimensions, pad the smaller shape with 1s on the left.
2. Dimensions with size 1 are stretched to match the other array.
3. If sizes don't match and neither is 1 → **error**.

### Examples:
```
(3,)          → (1, 3) → (4, 3)   ✓  broadcasts with (4, 3)
(4, 1)        → (4, 3)             ✓  broadcasts with (1, 3) or (3,)
(4, 3) + (3,) → ok, result (4, 3)
(4, 3) + (4,) → ERROR (4 != 3 and neither is 1)
```

Broadcasting is critical in AI/ML for:
- Adding bias vectors to batched outputs
- Normalizing data along specific axes
- Computing distance matrices efficiently

In [ ]:
import numpy as np

# Scalar broadcast
arr = np.array([1, 2, 3, 4])
print('arr + 10:', arr + 10)       # 10 broadcasts to shape (4,)
print('arr * 2.5:', arr * 2.5)

In [ ]:
# 1D broadcast with 2D
mat = np.array([[1, 2, 3],
                [4, 5, 6],
                [7, 8, 9]])  # shape (3, 3)

row_vec = np.array([10, 20, 30])   # shape (3,) -> broadcast as (1,3) -> (3,3)
col_vec = np.array([[10],          # shape (3,1) -> broadcast as (3,3)
                    [20],
                    [30]])

print('mat + row_vec:\n', mat + row_vec)
print('mat + col_vec:\n', mat + col_vec)

In [ ]:
# Visualizing broadcast step by step
import numpy as np

a = np.array([[1], [2], [3]])  # shape (3, 1)
b = np.array([10, 20, 30])    # shape (3,) -> treated as (1, 3)

print('a shape:', a.shape)     # (3, 1)
print('b shape:', b.shape)     # (3,)
result = a + b
print('a + b shape:', result.shape)  # (3, 3)
print(result)

In [ ]:
# Deep Learning: Adding bias to batched outputs
# batch_output: (batch_size=32, output_features=10)
batch_output = np.random.randn(32, 10)
bias = np.array([0.1, 0.2, 0.3, 0.4, 0.5,
                 0.6, 0.7, 0.8, 0.9, 1.0])  # shape (10,)

# bias broadcasts: (10,) -> (1, 10) -> (32, 10)
output_with_bias = batch_output + bias
print('Output + bias shape:', output_with_bias.shape)

In [ ]:
# Computing pairwise squared distances (no loops!)
# x: (N, D) points in D-dimensional space
x = np.random.randn(5, 3)  # 5 points in 3D

# x[:, np.newaxis, :] -> (5, 1, 3)
# x[np.newaxis, :, :] -> (1, 5, 3)
# diff: (5, 5, 3)
diff = x[:, np.newaxis, :] - x[np.newaxis, :, :]
dist_sq = np.sum(diff**2, axis=-1)  # (5, 5)
print('Pairwise distance matrix shape:', dist_sq.shape)
print(np.round(dist_sq, 2))

In [ ]:
# np.broadcast_to — explicitly broadcast without copy
arr = np.array([1, 2, 3])
broadcast = np.broadcast_to(arr, (4, 3))
print('broadcast_to (4,3):\n', broadcast)
print('Is it a view?', broadcast.base is arr)  # True

In [ ]:
# np.broadcast_shapes — check resulting shape without computing
print(np.broadcast_shapes((3, 1), (1, 4)))    # (3, 4)
print(np.broadcast_shapes((2, 3), (3,)))      # (2, 3)
print(np.broadcast_shapes((1,), (5,), (3, 5))) # (3, 5)

In [ ]:
# EXCEPTION: Incompatible shapes
a = np.ones((3, 4))
b = np.ones((3,))   # shape mismatch: 4 != 3 and neither is 1
try:
    c = a + b
except ValueError as e:
    print('ValueError:', e)

# Fine: (3, 1) + (3,) -> (3, 3)
a = np.ones((3, 1))
b = np.ones((3,))
print('(3,1) + (3,) shape:', (a + b).shape)